<a href="https://colab.research.google.com/github/f24607007-debug/PFAI-LAB/blob/main/LAB_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import jax
import jax.numpy as jnp
import pandas as pd
import seaborn as sns
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = sns.load_dataset('titanic')
df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']].dropna()

df['sex'] = df['sex'].map({'male': 0, 'female': 1})

# Separating features (X) and target (y)
X = df.drop('survived', axis=1).values
y = df['survived'].values

# Standardizing the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Split and convert to JAX arrays
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
X_train, y_train = jnp.array(X_train), jnp.array(y_train)
X_test, y_test = jnp.array(X_test), jnp.array(y_test)

print("Step 1 Complete: Data loaded and preprocessed without errors.")

Step 1 Complete: Data loaded and preprocessed without errors.


In [35]:
def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def predict(params, x):
    w, b = params
    return sigmoid(jnp.dot(x, w) + b)

def loss_fn(params, X, y):
    # Vectorized predictions across the whole batch
    preds = jax.vmap(predict, in_axes=(None, 0))(params, X)
    # Cross-entropy loss formula
    return -jnp.mean(y * jnp.log(preds + 1e-7) + (1 - y) * jnp.log(1 - preds + 1e-7))

In [36]:
key = jax.random.PRNGKey(42)
params = (jax.random.normal(key, (6,)), 0.0)

@jax.jit
def update(params, X, y, lr=0.1):
    grads = jax.grad(loss_fn)(params, X, y)
    return jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)

print("--- Step 3: Training Loop ---")
for epoch in range(101):
    params = update(params, X_train, y_train)
    if epoch % 20 == 0:
        current_loss = loss_fn(params, X_train, y_train)
        print(f"Epoch {epoch}: Loss = {current_loss:.4f}")


# Defining  a version of the update function WITHOUT @jax.jit for checking
def update_no_jit(params, X, y, lr=0.1):
    grads = jax.grad(loss_fn)(params, X, y)
    return jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)

print("\n--- Training Speed Evidence ---")

# Time Training WITHOUT JIT
start_no_jit = time.time()
for _ in range(10):
    params = update_no_jit(params, X_train, y_train)
time_no_jit = time.time() - start_no_jit
print(f"10 Epochs WITHOUT JIT: {time_no_jit:.4f}s")

# Time Training WITH JIT
update(params, X_train, y_train)
start_jit = time.time()
for _ in range(10):
    params = update(params, X_train, y_train)
time_jit = time.time() - start_jit
print(f"10 Epochs WITH JIT:    {time_jit:.4f}s")

print(f"Speedup Factor: {time_no_jit / time_jit:.2f}x faster with JIT")

--- Step 3: Training Loop ---
Epoch 0: Loss = 0.5955
Epoch 20: Loss = 0.5173
Epoch 40: Loss = 0.4842
Epoch 60: Loss = 0.4670
Epoch 80: Loss = 0.4569
Epoch 100: Loss = 0.4503

--- Training Speed Evidence ---
10 Epochs WITHOUT JIT: 0.1891s
10 Epochs WITH JIT:    0.0008s
Speedup Factor: 232.82x faster with JIT


In [37]:
# Combining jit with vmap for maximum speed
fast_batch_predict = jax.jit(jax.vmap(predict, in_axes=(None, 0)))

print("\n--- Step 4: Speed Comparison ---")
#  Standard Loop
start = time.time()
_ = [predict(params, x) for x in X_test]
print(f"Manual Loop Time: {time.time() - start:.4f}s")

#  Optimized
start = time.time()
_ = fast_batch_predict(params, X_test).block_until_ready()
print(f"JIT + VMAP Time:   {time.time() - start:.4f}s")

# Calculating Accuracy
final_probs = fast_batch_predict(params, X_test)
accuracy = jnp.mean((final_probs > 0.5) == y_test)
print(f"\nFinal Test Accuracy: {accuracy:.2%}")


--- Step 4: Speed Comparison ---
Manual Loop Time: 0.0290s
JIT + VMAP Time:   0.1747s

Final Test Accuracy: 75.52%
